# Chapter 02 — Micrograd Value Class (Simplified)

Every modern neural network learns through backpropagation — the algorithm that computes how much each weight contributed to the model's error. But most practitioners treat it as a black box. In this chapter, you will build an automatic differentiation engine from scratch, giving you deep intuition about how neural networks actually learn.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

# Demo: single gradient computation
a = Value(2.0)
b = Value(3.0)
c = a * b  # c = 6.0
c.backward()
print(f"c = a * b = {c.data}")
print(f"dc/da = {a.grad}")  # Expected: 3.0 (= b.data)
print(f"dc/db = {b.grad}")  # Expected: 2.0 (= a.data)

# Training loop simulation: gradients change every step
import random
random.seed(42)
print(f"\n{'Step':>4s} | {'a':>5s} | {'b':>5s} | {'c=a*b':>7s} | {'dc/da':>6s} | {'dc/db':>6s}")
print("-" * 48)
for step in range(5):
    a = Value(round(random.uniform(1.0, 5.0), 2))
    b = Value(round(random.uniform(1.0, 5.0), 2))
    c = a * b
    c.backward()
    print(f"{step:>4d} | {a.data:>5.2f} | {b.data:>5.2f} | {c.data:>7.2f} | {a.grad:>6.2f} | {b.grad:>6.2f}")
print("\nNotice: dc/da always equals b, dc/db always equals a — but values change each step!")

---

**Course:** Aprenda LLM | **Chapter 02**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.